# RLOO & GRPO -> Dr. GRPO

**Central question:** We already know REINFORCE and PPO. What problem does each of these three methods try to solve, and what does each change in the gradient?

## 1. REINFORCE → RLOO: Estimate a baseline without a critic

**Problem.** For `2 + 2 = ?`, the model generates `abcdef4` and receives reward 1. REINFORCE reinforces the sampled sequence:

$$
g = R\nabla_\theta\log\pi_\theta(y\mid x).
$$

But reward 1 alone does not tell us how well this completion performed relative to **the policy's other possible answers** to the **same prompt**. Can we estimate a prompt-specific baseline without training a value model?

>See? back to the high variance issue original REINFORCE paper faces 😂. **History just repeats itself.**

**Idea:** Sample several independent completions for one prompt and use the *other* completions as the baseline.

| Completion | Reward | Mean reward of *other* completions | RLOO advantage |
|---|---:|---:|---:|
| A1: `abcdef4` | 1 | 0 | **+1** |
| A2: `abcdef6` | 0 | 0.5 | **−0.5** |
| A3: `abcdef10` | 0 | 0.5 | **−0.5** |

$$
\boxed{A_i^{\mathrm{RLOO}}=R_i-\frac{1}{G-1}\sum_{j\ne i}R_j}
$$

So the sampled gradient contributions are proportional to:

$$
+1\,\nabla\log\pi(A1\mid x)
-0.5\,\nabla\log\pi(A2\mid x)
-0.5\,\nabla\log\pi(A3\mid x).
$$

**Why leave one out?** A baseline must not depend on the current sampled action (conditional on the prompt) to preserve the **unbiased REINFORCE estimator**. Other independently sampled completions satisfy this requirement.

A useful identity:

$$
\boxed{A_i^{\mathrm{RLOO}}=\frac{G}{G-1}(R_i-\bar R)}
$$

Thus, subtracting the *full group mean* produces a fixed rescaling of RLOO for a fixed group size, but its uncorrected estimator has a corresponding $\frac{G-1}{G}$ scale factor in expectation.

## 2. RLOO → GRPO: Normalize group differences and constrain updates

**Problem.** Different prompts can have very different **reward spreads**. 
>Should a group with larger reward differences -> produce proportionally larger updates?

**GRPO's choice:** Standardize rewards *within each prompt's group*:

$$
\boxed{A_i^{\mathrm{GRPO}}=\frac{R_i-\bar R}{\sigma_R}},
\qquad
\sigma_R=\sqrt{\frac1G\sum_i(R_i-\bar R)^2}.
$$

For our original group $R=[1,0,0]$:

$$
A^{\mathrm{RLOO}}=[1,-0.5,-0.5],\qquad
A^{\mathrm{GRPO}}=[1.414,-0.707,-0.707].
$$

Within this group the relative weights are unchanged. To see the *real* difference, change the reward spread:

| Rewards | RLOO advantages | GRPO advantages |
|---|---|---|
| $[1,0.5,0]$ | $[0.75,0,-0.75]$ | $[1.225,0,-1.225]$ |
| $[0.51,0.50,0.49]$ | $[0.015,0,-0.015]$ | $[1.225,0,-1.225]$ |

**Observation:** RLOO preserves the difference in reward magnitude; GRPO removes it. This can help when reward scales are inconsistent, but it can also amplify tiny, potentially noisy reward differences.

**GRPO is not just normalized RLOO.** Original GRPO also uses:

- **PPO-style token importance ratios and clipping** to constrain policy updates (without training a critic).
- **Per-response length normalization**: the original objective averages token-level surrogate terms over the $T_i$ generated tokens.

Writing $\ell_{i,t}$ for a token's PPO-style clipped surrogate contribution, the essential structure is:

$$
\boxed{J_{\mathrm{GRPO}}\propto\frac1G\sum_i\frac1{T_i}\sum_{t=1}^{T_i}\ell_{i,t}\!\left(A_i^{\mathrm{GRPO}}\right)}
$$

This leaves two questions: *Why divide by group standard deviation? And why divide by actual response length?*

## 3. GRPO → Dr. GRPO: Remove two unintended weightings

### Problem A — Group-std weighting (question-level bias)

The table above shows that $[0.51,0.50,0.49]$ and $[1,0.5,0]$ produce identical normalized advantages. In effect, each prompt's learning signal is multiplied by $1/\sigma_R$.

This makes sense if: 
>Rewards for different prompts use different scoring scales.

But not so much if:
>Rewards for all prompts are **already on the same scale**!

For binary rewards with a group size of 10:

- 1 correct / 9 incorrect: $\sigma_R=0.3$.
- 5 correct / 5 incorrect: $\sigma_R=0.5$.

The lower-variance group gets a larger scaling factor. This may be intentional in some settings, but it is **not required** by the expected-reward policy-gradient objective. 

**Dr. GRPO's first change:**

$$
\boxed{A_i^{\mathrm{Dr}}=R_i-\bar R}\qquad\text{(remove division by $\sigma_R$)}.
$$

### Problem B — Length weighting (response-level bias)

For a full autoregressive completion:

$$
\nabla\log\pi_\theta(y\mid x)=\sum_{t=1}^{T}\nabla\log\pi_\theta(y_t\mid x,y_{<t}).
$$

**It is a sum, not an average.** Dividing by the sampled length $T_i$ changes each token's coefficient from $A_i$ to $A_i/T_i$.

(Read next section to understand why GRPO's potential reasoning for dividing length)

Use two groups with the same binary reward pattern, each having one correct answer:

| Group | Correct answer | Length | GRPO advantage | Per-token coefficient $A/T$ |
|---|---|---:|---:|---:|
| Q1 | A1 (`abcdef4`) | 10 | +1.414 | **+0.14142** |
| Q2 | B3 (correct long answer) | 500 | +1.414 | **+0.00283** |

The correct 10-token answer has a **50× larger coefficient per token** than the correct 500-token answer. Likewise, longer incorrect answers receive smaller-magnitude negative coefficients per token. This is *not* a proof that the total gradient norm differs by 50×: token-gradient vectors still have to be summed.

Length averaging is intuitive if we want each response's *average token loss* to count equally. But it is not neutral if our target is the original **expected sequence reward**.

**Dr. GRPO's second change:** Replace each sampled $T_i$ with a *fixed* constant $L$ (e.g., generation budget $L=512$):

$$
\boxed{J_{\mathrm{Dr}}\propto\frac1{GL}\sum_i\sum_{t=1}^{T_i}\ell_{i,t}\!\left(R_i-\bar R\right)}
$$

For Q1-A1 and Q2-B3, the **per-token coefficient** is now identical:

$$
\frac{2/3}{512}\approx0.001302.
$$

This does **not** force their total gradients to be equal; the long response still has more token-gradient terms. The fixed denominator avoids *additionally* reweighting each response by its realized length.

## 4. The punchline: Did we return to RLOO?

**Almost, for the on-policy gradient—not for the complete training algorithm.** With fixed $G$, no active clipping, and the same sampled group:

$$
g_{\mathrm{Dr}}=\frac{G-1}{GL}\,g_{\mathrm{RLOO}}.
$$

Thus their gradients point in the same direction, differing by an overall scale. However, **Dr. GRPO retains PPO-style clipped optimization**, while original RLOO is REINFORCE-style. The two are not identical off-policy or when clipping matters.

| Method | Group advantage | Response aggregation | Policy update |
|---|---|---|---|
| **RLOO** | Leave-one-out mean | Sum of token log-prob gradients | REINFORCE-style |
| **GRPO** | $(R_i-\bar R)/\sigma_R$ | Divide each response by its own $T_i$ | PPO-style clipping |
| **Dr. GRPO** | $R_i-\bar R$ | Divide by fixed $L$ | PPO-style clipping |

**Mental model:** RLOO solves *baseline estimation without a critic*. GRPO adds *group-wise scale normalization* and *PPO-style constrained updates* (plus length averaging). Dr. GRPO challenges two normalizations that can inadvertently reweight prompts and completions.

**Lesson:** A normalization is not necessarily mathematically harmless. Ask whether it only changes the **global gradient scale**, or also changes the **relative weight of questions, responses, or tokens**. None of these algorithms solves token-level credit assignment: `abcdef4` still receives one sequence-level advantage.


# Why GRPO Normalizes by Response Length — A Reasonable Idea with a Hidden Assumption

## 1. The Original Intuition: Why Divide by Length?

Suppose two completions receive the same positive advantage:

- Completion A: 10 tokens
- Completion B: 100 tokens

Assume each token contributes a gradient of +1 along the same parameter direction.

| Completion | Sum of gradients | Mean of gradients |
|---|---:|---:|
| A (10 tokens) | 10 | 1 |
| B (100 tokens) | 100 | 1 |

Without length normalization, the longer completion contributes **10× more gradient**, simply because it contains more tokens.

A natural idea is:

> Each completion should count as one training example. Why should a longer completion automatically receive more weight?

Therefore, normalize by response length:

$$
g_{\text{mean}}=\frac{1}{T}\sum_{t=1}^{T}g_t
$$

This seems reasonable because:

- It prevents gradient magnitude from growing automatically with sequence length when token gradients are aligned.
- It gives completions a more comparable per-token average scale.
- It resembles the familiar practice of averaging token losses during pretraining.

**Historical nuance:** The original GRPO formulation inherited response-length averaging from earlier LLM PPO formulations. The connection to pretraining conventions is a plausible explanation, not a proven account of the authors' intentions.

## 2. The Hidden Assumption: More Tokens ≠ Proportionally Larger Gradient

The previous intuition implicitly assumes token gradients accumulate in roughly the same direction.

But gradients are **vectors**, and different tokens can push shared parameters in opposite directions.

Consider a single shared parameter \(\theta\).

Every token has a nonzero gradient with respect to \(\theta\).

**Completion A: 10 tokens**

- 6 tokens contribute +0.5 each.
- 4 tokens contribute −0.5 each.

$$
\sum_tg_t=6(0.5)-4(0.5)=1
$$

**Completion B: 100 tokens**

- 51 tokens contribute +0.5 each.
- 49 tokens contribute −0.5 each.

$$
\sum_tg_t=51(0.5)-49(0.5)=1
$$

Every token participates in backpropagation. No token has zero gradient.

Now compare:

| Completion | Gradient sum | Length-normalized gradient |
|---|---:|---:|
| A (10 tokens) | 1 | 0.1 |
| B (100 tokens) | 1 | 0.01 |

**Before normalization:** Both completions contribute the same gradient along $\theta$.

**After normalization:** Completion A contributes 10× more than Completion B.

Length normalization has introduced a difference that did not exist in the original sequence gradients.

This example is mathematically possible because token gradients can cancel. It illustrates a failure of the assumption, not a claim that all real LLM gradients behave this way.

## 3. The Fundamental Difference: Pretraining vs. Policy Gradient

For sequence-level reinforcement learning, the objective is:

$$
J(\theta)=\mathbb E_{y\sim\pi_\theta}[R(y)]
$$

The REINFORCE gradient naturally contains:

$$
\nabla_\theta J
=
\mathbb E\left[
A\sum_{t=1}^{T}
\nabla_\theta\log\pi_\theta(y_t\mid y_{<t},x)
\right]
$$

Notice: **SUM, not MEAN.**

The original GRPO formulation instead uses a per-response averaging factor:

$$
\frac{A}{T}
\sum_{t=1}^{T}
\nabla_\theta\log\pi_\theta(y_t\mid y_{<t},x)
$$

This introduces an additional, length-dependent weighting of sequence gradients.

That weighting **may be useful** for optimization, but it is **not a mathematical consequence** of maximizing expected sequence reward.

Dr. GRPO addresses this by replacing the response-dependent denominator with a fixed normalization constant.

## 4. The Mental Model to Remember

**The reasonable intuition:**

> Longer sequences contain more token-level loss terms. Averaging them might prevent longer responses from dominating training.

**The hidden assumption:**

> More tokens imply proportionally larger accumulated gradients.

**The overlooked fact:**

> Token gradients are vectors. They can reinforce or cancel each other. Sequence length alone does not determine the magnitude of their sum.

$$
\boxed{
\text{More tokens}
\;\not\Rightarrow\;
\text{Proportionally larger gradient}
}
$$

Fundamentally, credit assignment issue.